In [ ]:
!pip install torchinfo torchmetrics

In [ ]:
def is_running_in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_running_in_colab():
    %cd /content/drive/MyDrive/03_deeplob_replication/

import pandas as pd
import numpy as np
import polars as pl
from polars import col as c

import torch
from  torch.utils.data import Dataset, DataLoader
import torch.nn.functional as f
import torch.nn as nn
from torchinfo import summary
from torchmetrics import Accuracy, Precision, Recall
from torchmetrics.classification import MulticlassF1Score

from sklearn.metrics import confusion_matrix

import math

from src.model.model_set_up import DeepLob
from src.configs.configs import BASE_DIR, SRC_DIR, DATA_DIR
from src.create_dataset.create_dataset import load_data, data_processing,DPDataset
from src.train_and_evaluation.train_and_evaluate import train_and_evaluate, test_model

In [ ]:
train, test = load_data()
X_train, y_train = data_processing(train)
X_test, y_test = data_processing(test)
X_val = X_train[ int( math.floor( 0.8 * X_train.shape[0] )   ): ]
y_val = y_train[ int( math.floor( 0.8 * y_train.shape[0] )   ): ]
X_train = X_train[0: int( math.floor( 0.8 * X_train.shape[0] )   )]
y_train = y_train[0: int( math.floor( 0.8 * y_train.shape[0] )   )]
print(X_train.shape)
print(y_train.shape)

In [ ]:
batch_size =64
h = 0
num_classes =3
epochs = 50

dataset_train = DPDataset(X_train, y_train, h)
train_loader = DataLoader(dataset_train, batch_size, shuffle = True)

dataset_val = DPDataset(X_val, y_val, h)
val_loader = DataLoader(dataset_val, batch_size, shuffle = False)


dataset_test = DPDataset(X_test, y_test, h)
test_loader = DataLoader(dataset_test, batch_size, shuffle = False)

In [ ]:
model = DeepLob(y_len = num_classes)
print(model)
summary(model,(1,1,100,40))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')
device= 'cuda' if torch.cuda.is_available() else 'cpu'

best_path=BASE_DIR/'model_trained/best_model.pt'
train_and_evaluate(model, train_loader, val_loader, criterion,
    optimizer, n_epochs=epochs, device='cuda', scheduler=None,
                 early_stop=False, patience=5, path = best_path)
best_model=torch.load(best_path, weights_only=True)
model.load_state_dict(best_model)
test_model(model, test_loader, criterion, device=device)